In [1]:
# -*- coding: utf-8 -*-
"""
Stratified sampling with per-stratum MOE control (±10 pp @ 90% CI, worst-case p=0.5, with FPC)

- 8 strata: Y{0/1}_B{0/1}_A{0/1}
- Guarantees the MOE target per stratum (no overall cap)
- Reproducible draws per stratum (stable seed)
- Minimal output columns in sample:
    html_url, full_name, language, stratum, YAML_pred, Build_pred, AT_pred

Required dataset columns:
- instru_t_ci_signal       -> CI YAML signal (bool/0/1)
- instru_t_signal_config   -> Build/Gradle instrumentation signal (bool/0/1)
- Intru_test               -> androidTest present (bool/0/1)
Optional identifiers:
- full_name, html_url, language
"""

import os
import math
import hashlib
import numpy as np
import pandas as pd

# ---------- CONFIG ----------
DATA_PATH  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Per-stratum design target (no overall total cap)
MOE_TARGET = 0.10   # ±10 percentage points
CONF_LEVEL = 0.90   # 90% CI

# Optional per-stratum minimum "top-up" sample sizes, e.g., {"Y1_B0_A1": 31}
TOP_UP_STRATA = {}

# Reproducible sampling base seed
#BASE_SEED = 20250827 # fixing the issued
BASE_SEED = 20250828 # final check

# ---------- UTILS ----------
def to_bin(s: pd.Series) -> pd.Series:
    """Convert common truthy values to {0,1} robustly."""
    if s.dtype == bool:
        return s.astype(int)
    if s.dtype.kind in "biufc":
        return (s.fillna(0) != 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(["1","true","t","yes","y"]).astype(int)

def z_for(conf: float) -> float:
    """Return z critical (two-sided) for common confidence levels."""
    if abs(conf - 0.95) < 1e-12:
        return 1.96
    if abs(conf - 0.90) < 1e-12:
        return 1.645
    if abs(conf - 0.99) < 1e-12:
        return 2.576
    return 1.96

def n_for_moe(N: int, e: float, p: float = 0.5, conf: float = 0.95) -> int:
    """
    Minimum n to achieve half-width e for proportion p at 'conf' with finite-population correction.
    n = [z^2 * p(1-p) * N] / [e^2*(N-1) + z^2 * p(1-p)]
    """
    if N <= 0:
        return 0
    z = z_for(conf)
    num = (z**2) * p * (1 - p) * N
    den = (e**2) * (N - 1) + (z**2) * p * (1 - p)
    n = int(math.ceil(num / den))
    return max(0, min(n, N))

def realized_moe_p05(N: int, n: int, conf: float = 0.95) -> float:
    """Realized half-width at p=0.5 with FPC for given (N, n)."""
    z = z_for(conf)
    if N <= 1 or n <= 0:
        return float("inf")
    se = math.sqrt(0.25 / n) * math.sqrt((N - n) / (N - 1))
    return z * se

def stable_seed(label: str, base_seed: int = BASE_SEED) -> int:
    """Deterministic seed from a label + base seed."""
    return (int(hashlib.md5(label.encode("utf-8")).hexdigest()[:8], 16) ^ base_seed) & 0x7FFFFFFF

# ---------- LOAD & PREP ----------
df = pd.read_csv(DATA_PATH, low_memory=False)

req_cols = {
    "YAML_pred":  "instru_t_ci_signal",
    "Build_pred": "instru_t_signal_config",
    "AT_pred":    "Instru_test",
}
missing = [v for v in req_cols.values() if v not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["YAML_pred"]  = to_bin(df[req_cols["YAML_pred"]])
df["Build_pred"] = to_bin(df[req_cols["Build_pred"]])
df["AT_pred"]    = to_bin(df[req_cols["AT_pred"]])

# optional identifiers
if "html_url" not in df.columns:
    df["html_url"] = ""
if "full_name" not in df.columns:
    df["full_name"] = ""
# NEW: language passthrough (ensure column exists; keep strings)
if "language" not in df.columns:
    df["language"] = ""
else:
    df["language"] = df["language"].astype(str).fillna("")

# 8 strata labels
df["stratum"] = df.apply(
    lambda r: f"Y{int(r['YAML_pred'])}_B{int(r['Build_pred'])}_A{int(r['AT_pred'])}", axis=1
)

# ---------- COHORT SIZES ----------
cohort = (
    df.groupby("stratum", dropna=False)
      .size()
      .rename("N")
      .reset_index()
      .sort_values("stratum")
      .reset_index(drop=True)
)

# Ensure all 8 strata rows exist
all_strata = [f"Y{y}_B{b}_A{a}" for y in (0,1) for b in (0,1) for a in (0,1)]
if set(all_strata) - set(cohort["stratum"]):
    missing_rows = pd.DataFrame({"stratum": list(set(all_strata) - set(cohort["stratum"]))})
    missing_rows["N"] = 0
    cohort = pd.concat([cohort, missing_rows], ignore_index=True).sort_values("stratum").reset_index(drop=True)

# Minimum n per stratum to guarantee ±10pp @ 90%
cohort["alloc"] = cohort["N"].apply(lambda N: n_for_moe(N, e=MOE_TARGET, p=0.5, conf=CONF_LEVEL))

# Optional top-ups
for s, n_min in TOP_UP_STRATA.items():
    mask = cohort["stratum"] == s
    if mask.any():
        cohort.loc[mask, "alloc"] = np.maximum(cohort.loc[mask, "alloc"].astype(int), int(n_min))

# Safety cap by available N
cohort["alloc"] = cohort[["alloc", "N"]].min(axis=1).astype(int)

# Realized MOE check (p=0.5)
cohort["moe_p05_used"] = cohort.apply(
    lambda r: round(realized_moe_p05(int(r["N"]), int(r["alloc"]), conf=CONF_LEVEL), 4), axis=1
)

# ---------- DRAW SAMPLE (REPRODUCIBLE) ----------
# NEW: include 'language' in the output columns
cols_out = ["html_url", "full_name", "language", "stratum", "YAML_pred", "Build_pred", "AT_pred"]
samples = []
for _, row in cohort.iterrows():
    s = row["stratum"]
    k = int(row["alloc"])
    if k <= 0:
        continue
    pool = df[df["stratum"] == s][cols_out]
    if len(pool) == 0:
        continue
    take = min(k, len(pool))
    rs = np.random.RandomState(stable_seed(s))
    smp = pool.sample(n=take, replace=False, random_state=rs).copy()
    samples.append(smp)

sampled_df = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=cols_out)

# ---------- PLANS & COHORT TABLE ----------
out_plan = cohort.copy()
out_plan.insert(1, "ci_used", f"{int(CONF_LEVEL*100)}%")
out_plan["weight"] = np.where(out_plan["alloc"] > 0, out_plan["N"] / out_plan["alloc"], np.nan)

# Compact population vs sample table (+ totals)
cohort_table = (
    cohort.loc[:, ["stratum", "N", "alloc"]]
          .rename(columns={"N": "population", "alloc": "sample"})
          .sort_values("stratum", ignore_index=True)
)
totals = pd.DataFrame([{
    "stratum": "TOTAL",
    "population": int(cohort_table["population"].sum()),
    "sample":     int(cohort_table["sample"].sum()),
}])
cohort_table = pd.concat([cohort_table, totals], ignore_index=True)

# Optional coverage rate
cohort_table["sample_rate"] = np.where(
    cohort_table["stratum"] != "TOTAL",
    cohort_table["sample"] / cohort_table["population"].replace(0, np.nan),
    cohort_table.loc[cohort_table["stratum"] == "TOTAL","sample"].values
    / cohort_table.loc[cohort_table["stratum"] == "TOTAL","population"].values
)

# ---------- SAVE ----------
plan_path          = os.path.join(OUTPUT_DIR, "allocation_plan_moe10.csv")
sample_path        = os.path.join(OUTPUT_DIR, "stratified_sample_moe10.csv")
cohort_table_path  = os.path.join(OUTPUT_DIR, "cohort_table.csv")

out_plan.to_csv(plan_path, index=False, encoding="utf-8")
sampled_df.to_csv(sample_path, index=False, encoding="utf-8")
cohort_table.to_csv(cohort_table_path, index=False, encoding="utf-8")

# ---------- SUMMARY ----------
print(f"Confidence level: {int(CONF_LEVEL*100)}%")
print(f"Target MOE (half-width): ±{int(MOE_TARGET*100)} pp at worst-case p=0.5")
print(f"Total population (N): {int(out_plan['N'].sum())}")
print(f"Total sampled (sum of per-stratum mins): {int(out_plan['alloc'].sum())}")

print("\nPer-stratum allocation (N, alloc, realized_moe@p=0.5, weight):")
print(out_plan[["stratum", "N", "alloc", "moe_p05_used", "weight"]])

print("\nCohort table (population vs. sample, with sample_rate):")
print(cohort_table)

print(f"\nSaved:\n  {plan_path}\n  {sample_path}\n  {cohort_table_path}")


Confidence level: 90%
Target MOE (half-width): ±10 pp at worst-case p=0.5
Total population (N): 4518
Total sampled (sum of per-stratum mins): 384

Per-stratum allocation (N, alloc, realized_moe@p=0.5, weight):
    stratum     N  alloc  moe_p05_used     weight
0  Y0_B0_A0  1926     66        0.0995  29.181818
1  Y0_B0_A1   214     52        0.0995   4.115385
2  Y0_B1_A0   733     63        0.0991  11.634921
3  Y0_B1_A1  1168     64        0.1000  18.250000
4  Y1_B0_A0    44     27        0.0995   1.629630
5  Y1_B0_A1    42     27        0.0957   1.555556
6  Y1_B1_A0    45     28        0.0966   1.607143
7  Y1_B1_A1   346     57        0.0997   6.070175

Cohort table (population vs. sample, with sample_rate):
    stratum  population  sample  sample_rate
0  Y0_B0_A0        1926      66     0.034268
1  Y0_B0_A1         214      52     0.242991
2  Y0_B1_A0         733      63     0.085948
3  Y0_B1_A1        1168      64     0.054795
4  Y1_B0_A0          44      27     0.613636
5  Y1_B0_A1  